In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print(" Ready")

 Ready


In [2]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph , START , END

class DemoState(TypedDict):
    results: Annotated[list,add]
def branch_a(state:DemoState)-> dict:
    return{"results":["from_a"]}

def branch_b(state: DemoState)-> dict:
    return{"results":["from_b"]}

def branch_c(state: DemoState)->dict:
    return{"results":["from_c"]}

graph=StateGraph(DemoState)
graph.add_node("a",branch_a)
graph.add_node("b",branch_b)
graph.add_node("c",branch_c)

graph.add_edge(START,"a")
graph.add_edge(START,"b")
graph.add_edge(START,"c")

graph.add_edge("a",END)
graph.add_edge("b",END)
graph.add_edge("c",END)

app= graph.compile()
result=app.invoke({"results":[]})
print("merged results:",result["results"])
assert set(result["results"])=={"from_a","from_b","from_c"}
print(" All three parallel branches' writes were merged, none overwrote another")

merged results: ['from_a', 'from_b', 'from_c']
 All three parallel branches' writes were merged, none overwrote another


In [3]:
class BrokenState(TypedDict):
    results:list

def broken_a(state:BrokenState)->dict:
    return{"results":["from_a"]}

def broken_b(state:BrokenState)->dict:
    return{"results":["from_b"]}

graph2=StateGraph(BrokenState)
graph2.add_node("a",broken_a)
graph2.add_node("b",broken_b)
graph2.add_edge(START,"a")
graph2.add_edge(START,"b")

graph2.add_edge("a",END)
graph2.add_edge("b",END)
try:
    app2=graph2.compile()
    result2=app2.invoke({"results":[]})
    print("Result:", result2["results"])
except Exception as e:
    print(f" {type(e).__name__}: {e}")
    print("\nThis is actually the SAFER failure mode — LangGraph detects the conflict")
    print("and refuses to guess, rather than silently dropping one branch's write.")

 InvalidUpdateError: At key 'results': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE

This is actually the SAFER failure mode — LangGraph detects the conflict
and refuses to guess, rather than silently dropping one branch's write.


In [4]:
from langgraph.types import Send
from typing import List

class FanOutState(TypedDict):
    queries:List[str]
    search_results:Annotated[list,add]

def dispatch(state:FanOutState):
    """Returns a list of sned onjects-one per query-which langGraph runs as genuinely parallel branches , each with its own isolated state slice """
    return[Send("search_one_node",{"query":q}) for q in state["queries"]]

def search_one_node(state:dict)->dict:
    # each parallel invocation gets ONLY {"query": q}, not the full FanOutState
    return {"search_results": [f"result_for_{state['query']}"]}

graph3 = StateGraph(FanOutState)
graph3.add_node("search_one_node", search_one_node)
graph3.add_conditional_edges(START, dispatch, ["search_one_node"])
graph3.add_edge("search_one_node", END)

app3 = graph3.compile()
result3 = app3.invoke({"queries": ["q1", "q2", "q3"], "search_results": []})
print("Fan-out results:", result3["search_results"])
assert len(result3["search_results"]) == 3
print("Send-based fan-out correctly ran 3 parallel branches and merged all 3 results")

Fan-out results: ['result_for_q1', 'result_for_q2', 'result_for_q3']
Send-based fan-out correctly ran 3 parallel branches and merged all 3 results


In [5]:
import random

def search_one_node_variable_delay(state: dict) -> dict:
    import time
    time.sleep(random.uniform(0.01, 0.1))  # simulate variable API latency
    return {"search_results": [f"result_for_{state['query']}"]}

graph4 = StateGraph(FanOutState)
graph4.add_node("search_one_node", search_one_node_variable_delay)
graph4.add_conditional_edges(START, dispatch, ["search_one_node"])
graph4.add_edge("search_one_node", END)
app4 = graph4.compile()

for trial in range(3):
    r = app4.invoke({"queries": ["q1", "q2", "q3"], "search_results": []})
    print(f"Trial {trial}:", r["search_results"])

print("\n Order is NOT guaranteed across trials — downstream agents (Data Extraction,")
print("Fact-Checking) must not assume search_results[0] corresponds to plan[0].")
print("They already don't — they read sub_question from each result dict itself, not by index.")

Trial 0: ['result_for_q1', 'result_for_q2', 'result_for_q3']
Trial 1: ['result_for_q1', 'result_for_q2', 'result_for_q3']
Trial 2: ['result_for_q1', 'result_for_q2', 'result_for_q3']

 Order is NOT guaranteed across trials — downstream agents (Data Extraction,
Fact-Checking) must not assume search_results[0] corresponds to plan[0].
They already don't — they read sub_question from each result dict itself, not by index.


In [6]:
import inspect
from agents.data_extraction_agent import extract_facts_from_result

source = inspect.getsource(extract_facts_from_result)
print(source)
print("\nConfirmed: this function takes ONE search_result dict and reads")
print("sub_question/answer/sources FROM that dict — it never indexes into a list")
print("by position, so merge order from parallel execution cannot corrupt it.")

def extract_facts_from_result(search_result: Dict) -> List[Dict]:
    sub_question = search_result.get("sub_question", "")
    answer = search_result.get("answer", "")
    sources = search_result.get("sources", [])

    if not answer.strip():
        return []

    try:
        raw_output = call_llm(_build_extraction_prompt(sub_question, answer))
        raw_facts = _parse_json_facts(raw_output)
    except Exception as e:
        log_agent_event(AGENT_NAME, "extraction_failed", sub_question=sub_question,
                        error=f"{type(e).__name__}: {e}")
        return []

    facts = [{"claim": t, "sub_question": sub_question, "sources": sources} for t in raw_facts]
    log_agent_event(AGENT_NAME, "extraction_done", sub_question=sub_question, facts_found=len(facts))
    return facts


Confirmed: this function takes ONE search_result dict and reads
sub_question/answer/sources FROM that dict — it never indexes into a list
by position, so merge order from parallel execution cannot

In [7]:
from agents.web_search_agent import search_one
from langgraph.types import Send

class RealFanOutState(TypedDict):
    plan: List[str]
    search_results: Annotated[list, add]

def real_dispatch(state: RealFanOutState):
    return [Send("real_search", {"query": q}) for q in state["plan"]]

def real_search_node(state: dict) -> dict:
    result = search_one(state["query"])
    return {"search_results": [result]}

graph5 = StateGraph(RealFanOutState)
graph5.add_node("real_search", real_search_node)
graph5.add_conditional_edges(START, real_dispatch, ["real_search"])
graph5.add_edge("real_search", END)
app5 = graph5.compile()

real_plan = [
    "Tesla 4680 battery energy density 2026",
    "CATL sodium-ion battery range",
    "BYD blade battery charging speed",
]

result5 = app5.invoke({"plan": real_plan, "search_results": []})
print(f"Got {len(result5['search_results'])} results from {len(real_plan)} parallel real searches")
for r in result5["search_results"]:
    print(f"[{r['status']}] {r['sub_question']}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


 Gemini unavailable (ClientError), falling back to Tavily...
 Gemini unavailable (ClientError), falling back to Tavily...
 Gemini unavailable (ClientError), falling back to Tavily...


2026-10-08 20:33:24,153 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3
2026-10-08 20:33:24,300 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3
2026-10-08 20:33:24,519 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


Got 3 results from 3 parallel real searches
[ok] Tesla 4680 battery energy density 2026
[ok] CATL sodium-ion battery range
[ok] BYD blade battery charging speed


In [ ]:
doc = """# State Management Deep-Dive (Day 16)

## Verified
1. `Annotated[list, add]` correctly merges writes from parallel branches run in
   the same LangGraph superstep — confirmed with 3 synthetic branches, all 3
   writes present in the final state, none lost or overwritten.
2. A plain `list` field (no reducer) causes LangGraph to raise an error on
   concurrent writes rather than silently dropping one — this is the SAFE
   failure mode, not a bug. It forces the developer to declare a reducer
   rather than getting silent data loss.
3. LangGraph's `Send` API is the idiomatic fan-out primitive — dispatches N
   parallel branches, each with its own isolated state slice (not the full
   parent state), results merge back via the reducer. Verified with both
   synthetic and REAL search_one calls (3 real parallel searches, all 3
   results correctly merged).
4. Merge ORDER is not guaranteed across parallel branches. Verified this does
   NOT break anything downstream: data_extraction_agent and all subsequent
   agents read fields (sub_question, answer, sources) from each result dict
   directly, never by list position/index. This was an accidental but
   important design correctness from Day 9 onward.

## Decision for Day 17
Use `Send` (LangGraph's native fan-out), not Day 15's ThreadPoolExecutor
approach, for the actual web_search_node implementation. Reasoning:
- Send is the idiomatic LangGraph mechanism, integrates with the existing
  reducer-based state schema with zero extra glue code.
- ThreadPoolExecutor (Day 15) proved the SPEEDUP is real and worth pursuing,
  but Send achieves the same parallelism while staying inside LangGraph's
  state/checkpointing/tracing model, rather than doing ad-hoc threading
  inside a single node that LangGraph can't see into.
- Revising Day 15's conclusion: Send over ThreadPoolExecutor for the final
  implementation, now that this deep-dive confirmed Send's mechanics are
  correct, safe, and already proven against real API calls.

## Remaining risk carried into Day 17
Send-based fan-out will still make N simultaneous API calls, same as Day 15's
ThreadPoolExecutor did — the intermittent rate-limit failures found on Day 15
are not specific to ThreadPoolExecutor and will still need Day 18's retry work
regardless of which mechanism is used.
"""

with open("docs/state_management_deep_dive.md", "w", encoding="utf-8") as f:
    f.write(doc)
print(" docs/state_management_deep_dive.md written")